# 🌌 Gemini Embedding 2: Guía Completa y Práctica con Python

Bienvenido a este cuaderno interactivo de estudio. Este tutorial implementa en profundidad las capacidades de **Gemini Embedding 2** ( / ), el primer modelo de embeddings nativamente multimodal de Google, en combinación con **ChromaDB** para búsqueda vectorial cruzada.

Basado en la guía de DataCamp por *Vaibhav Mehra* y ampliado con mejores prácticas de ingeniería de software.

---
### 📚 Contenido del Tutorial
1. **¿Qué son los Embeddings?** Espacios latentes y similitud coseno.
2. **Novedades de Gemini Embedding 2:** Arquitectura unificada vs. modelos *Two-Tower*.
3. **Configuración del Entorno:** SDK  y API Key.
4. **Tu Primer Embedding y MRL:** Reducción de dimensiones Matrioshka (3072 $	o$ 768).
5. **Prefijos de Tarea (*Task Prefixes*):** Búsqueda asimétrica y tareas simétricas.
6. **El Comportamiento Crítico:** Vectores Fusionados (*Merged*) vs. Independientes (*Separate*).
7. **Construcción de la Base de Datos Vectorial:** ChromaDB con métrica coseno.
8. **Búsqueda Cruzada Multimodal:**
   - Texto $	o$ Imagen (comparativa con y sin prefijo)
   - Imagen $	o$ Imagen (Búsqueda visual)
   - Texto $	o$ Audio (Recuperación sonora y reproducción)
9. **Limitaciones y Consideraciones en Producción**

## 1. Fundamentos Teóricos

### ¿Qué es un Embedding?
Un **embedding** es una lista de números en coma flotante (un vector) que representa el significado semántico de un fragmento de contenido.

Podemos visualizarlo como una **coordenada en un mapa multidimensional**:
- Conceptos similares (por ejemplo, *"un cachorro jugando en el césped"* y *"un perro joven corriendo en el pasto"*) se posicionan muy cerca entre sí.
- Conceptos distantes (por ejemplo, *"declaración trimestral de impuestos"*) quedan muy alejados en el espacio.

### La Métrica: Similitud Coseno
La cercanía en este mapa de alta dimensión se mide calculando el ángulo entre vectores mediante el **coseno del ángulo** $	heta$:

96038	ext{Similitud Coseno}(\mathbf{u}, \mathbf{v}) = rac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\| \|\mathbf{v}\|} = \cos(	heta)96038

- **1.0**: Significado idéntico (mismo ángulo).
- **0.0**: Ortogonal (no relacionados semánticamente).
- **-1.0**: Completamente opuestos.

### ¿Por qué un Espacio Multimodal Compartido es Revolucionario?
Históricamente, los modelos de texto generaban un espacio vectorial y los de imágenes (como CLIP) otro. Comparar un texto con una imagen requería alineaciones post-hoc o arquitecturas de dos torres (*Two-Tower*).

**Gemini Embedding 2** proyecta **texto, imágenes, audio, video y PDFs dentro de la misma y única coordenada espacial**. Una foto de un perro, el audio de un ladrido y el texto *"perro ladrando"* caen en el mismo vecindario vectorial.

## 2. Configuración del Cliente SDK y Clave de API

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from google import genai
from google.genai import types

# Cargar variables de entorno desde .env si existe
load_dotenv()

api_key = os.environ.get("GEMINI_API_KEY")
if not api_key:
    print(
        "⚠️ ADVERTENCIA: GEMINI_API_KEY no detectada. Configúrala en .env o en el entorno para realizar llamadas reales."
    )
else:
    print("✓ GEMINI_API_KEY configurada correctamente.")

# Inicializar cliente de Google GenAI
client = genai.Client(api_key=api_key) if api_key else None
MODEL_NAME = "gemini-embedding-2"
DIMS = 768  # Dimensión Matryoshka recomendada

## 3. Tu Primer Embedding y Matryoshka Representation Learning (MRL)

Por defecto, Gemini Embedding 2 genera vectores de **3.072 dimensiones**.
Sin embargo, almacenar 3.072 dimensiones para millones de elementos es computacionalmente costoso.

Para resolver esto, el modelo está entrenado con **Matryoshka Representation Learning (MRL)**, similar a las muñecas rusas:
- Los primeros **768** números concentran la mayor parte de la información semántica fundamental.
- Los siguientes números (hasta 1.536 o 3.072) añaden detalles cada vez más finos.
- ¡Al truncar a 768 dimensiones ahorramos un **75% de almacenamiento** con una pérdida mínima de precisión!

In [ ]:
# Ejemplo: Comparación de dimensiones por defecto (3072) vs Matryoshka (768)
if client:
    # Vector completo (3072)
    full_res = client.models.embed_content(
        model=MODEL_NAME,
        contents="A robotic arm working in an automation assembly line",
    )
    full_vec = full_res.embeddings[0].values
    print(f"Dimensiones por defecto: {len(full_vec)}")

    # Vector truncado con MRL (768)
    mrl_res = client.models.embed_content(
        model=MODEL_NAME,
        contents="A robotic arm working in an automation assembly line",
        config=types.EmbedContentConfig(output_dimensionality=768),
    )
    mrl_vec = mrl_res.embeddings[0].values
    print(f"Dimensiones con MRL:      {len(mrl_vec)}")
else:
    print("Dimensiones por defecto esperadas: 3072 | Con MRL: 768")

## 4. Prefijos de Tarea (*Task Prefixes*)

Una misma frase debe representarse de manera distinta según el objetivo (¿es una pregunta buscando respuesta o un documento indexado?).

Gemini Embedding 2 **no usa el parámetro legacy **, sino que introduce el formato de prefijo dentro del texto:

| Caso de Uso | Formato de Consulta (*Query*) | Formato de Documento (*Document*) |
| :--- | :--- | :--- |
| **Búsqueda (*Search*)** |  |  (o ) |
| **Preguntas y Respuestas (QA)** |  |  |
| **Verificación (*Fact Checking*)** |  |  |
| **Código (*Code Retrieval*)** |  |  |
| **Clasificación / Clustering** |  | Idéntico a la consulta (Simétrico) |
| **Similitud Semántica** |  | Idéntico a la consulta (Simétrico) |

> ⚠️ **Regla Crítica de Google:** NO apliques prefijos de tarea al texto de entradas multimodales combinadas (por ejemplo, pie de foto + imagen), ya que degrada el rendimiento.

In [ ]:
from gemini_embedding.embeddings import format_task_text

# Consulta asimétrica de búsqueda
q = format_task_text("red sports car on highway", task="search result", is_query=True)
print("Query:", q)

# Documento asimétrico indexado
doc = format_task_text(
    "High performance vehicle details",
    task="search result",
    is_query=False,
    title="Sports Car Specs",
)
print("Document:", doc)

# Tarea simétrica de clasificación
cat = format_task_text("Technical support inquiry", task="classification")
print("Classification:", cat)

## 5. El Comportamiento Crítico: Entradas Fusionadas vs. Lotes Separados

Uno de los errores más frecuentes en Gemini Embedding 2:
- En modelos antiguos, pasar una lista de 5 elementos devolvía 5 embeddings.
- En Gemini Embedding 2, pasar  **combina ambos en un único vector conjunto** (entrada multimodal intercalada).
- Para procesar un lote y obtener **un vector por cada elemento**, se debe envolver cada elemento en su propio objeto .

In [ ]:
from gemini_embedding.sample_media import generate_sample_media

# Generar muestras sintéticas locales para pruebas
media_files = generate_sample_media("./media")
print("Archivos de prueba generados:", [p.name for p in media_files["images"]])

dog_img_path = (
    Path("./media/dog.jpg") if Path("./media/dog.jpg").exists() else Path("./media/dog.png")
)
if client and dog_img_path.exists():
    img_part = types.Part.from_bytes(data=dog_img_path.read_bytes(), mime_type="image/png")

    # CASO 1: Lista de partes -> 1 solo vector fusionado
    merged = client.models.embed_content(
        model=MODEL_NAME,
        contents=["A playful golden puppy on the grass", img_part],
    )
    print(f"Merged (partes juntas): {len(merged.embeddings)} embedding resultante")

    # CASO 2: Lista de objetos Content -> 1 vector individual por elemento
    separate = client.models.embed_content(
        model=MODEL_NAME,
        contents=[
            types.Content(parts=[types.Part.from_text(text="Texto independiente")]),
            types.Content(parts=[img_part]),
        ],
    )
    print(f"Separate (objetos Content): {len(separate.embeddings)} embeddings resultantes")
else:
    print("Ejecuta con GEMINI_API_KEY para verificar en vivo la respuesta del API.")

## 6. Configuración de la Base de Datos Vectorial (ChromaDB)

ChromaDB es una base de datos vectorial local y embebible.

> ⚠️ **CRÍTICO:** ChromaDB utiliza por defecto distancia euclidiana (L2). Los embeddings de Gemini están diseñados y normalizados para evaluarse con **distancia coseno**. Por eso es indispensable configurar .

In [ ]:
from gemini_embedding.vector_store import MultimodalVectorStore

# Inicializar almacén persistente con distancia coseno
vector_store = MultimodalVectorStore(persist_dir="./chroma_db", collection_name="multimodal_media")
print(f"Colección cargada: {vector_store.collection_name}")
print(f"Elementos indexados actualmente: {vector_store.count()}")

## 7. Indexación de Archivos Multimedia

Recorremos los archivos de  (imágenes PNG, audios WAV), calculamos sus embeddings respetando el límite de peticiones con , y los guardamos en ChromaDB con metadatos (, , ).

In [ ]:
from gemini_embedding.indexer import index_directory

if client:
    total = index_directory(
        media_dir="./media",
        client=client,
        vector_store=vector_store,
        rate_limit_delay=0.5,
        show_progress=True,
    )
    print(f"✓ Total de archivos indexados en ChromaDB: {total}")
else:
    print(
        "Para indexar tus archivos en vivo, ingresa tu GEMINI_API_KEY en .env y ejecuta esta celda."
    )

## 8. Experimentos de Búsqueda Cruzada Multimodal

### Experimento A: Búsqueda de Imágenes mediante Texto
Probamos consultar el catálogo de imágenes con una descripción en lenguaje natural y comparamos la similitud semántica con y sin prefijo de tarea.

In [ ]:
from gemini_embedding.search_engine import search_by_text
from gemini_embedding.visualizer import plot_image_results, render_results_table

query_text = "a red sports car parked on a paved street"

if client and vector_store.count() > 0:
    # Búsqueda con prefijo oficial de Google
    results_with_prefix = search_by_text(
        query_text, kind="image", use_prefix=True, vector_store=vector_store, client=client
    )
    print("--- Resultados CON prefijo de tarea ---")
    render_results_table(results_with_prefix)
    plot_image_results(results_with_prefix, show=True)

    # Búsqueda sin prefijo
    results_no_prefix = search_by_text(
        query_text, kind="image", use_prefix=False, vector_store=vector_store, client=client
    )
    print("--- Resultados SIN prefijo de tarea ---")
    render_results_table(results_no_prefix)
else:
    print("Índice vacío o API key no configurada.")

### Experimento B: Búsqueda Visual (Imagen a Imagen)
Utilizamos la imagen  para encontrar imágenes visual y semánticamente análogas sin usar ninguna palabra clave.

In [ ]:
from gemini_embedding.search_engine import search_by_image

query_img = Path("query.jpg") if Path("query.jpg").exists() else Path("media/query.jpg")

if client and query_img.exists() and vector_store.count() > 0:
    print(f"Buscando imágenes similares a: {query_img.name}")
    img_results = search_by_image(query_img, kind="image", vector_store=vector_store, client=client)
    render_results_table(img_results, title=f"Visual Search: {query_img.name}")
    plot_image_results(img_results, show=True)
else:
    print("Asegúrate de que query.jpg exista e indexa los medios para ver la búsqueda visual.")

### Experimento C: Búsqueda de Audio mediante Texto
Describimos un sonido con palabras (*"someone playing gentle piano chords"*) y recuperamos el clip de audio correspondiente para reproducirlo directamente en el cuaderno.

In [ ]:
from IPython.display import Audio

audio_query = "someone playing gentle harmonic piano notes"

if client and vector_store.count() > 0:
    audio_results = search_by_text(
        audio_query, kind="audio", n_results=1, vector_store=vector_store, client=client
    )
    render_results_table(audio_results, title=f"Audio Search: {audio_query}")
    if audio_results and Path(audio_results[0]["path"]).exists():
        matched_audio_path = audio_results[0]["path"]
        print(f"Reproduciendo archivo recuperado: {matched_audio_path}")
        display(Audio(filename=matched_audio_path))
else:
    print("Indexa los audios de muestra para probar la recuperación semántica de sonido.")

## 9. Limitaciones y Consideraciones Clave

1. **Entradas Cortas:** Video limitado a 120 segundos (máximo 32 frames), audio a 180 segundos, PDFs a 6 páginas (258 tokens por página para representación visual).
2. **Truncado Silencioso:** Peticiones que superan 8.192 tokens se truncan sin arrojar excepción. Monitorea los tokens al trabajar con documentos largos.
3. **El Video Ignora el Audio:** En archivos de video, la pista de sonido no se procesa. Si el audio es relevante, extráelo y embédelo por separado.
4. **Espacios Incompatibles:** Los vectores de  y  viven en espacios métricos distintos y **no se pueden comparar**; migrar exige reindexar.
5. **Modelo Propietario vía API:** Los datos viajan a la infraestructura de Google Cloud.

--- 
### 🎓 Conclusión
Gemini Embedding 2 unifica por primera vez la búsqueda semántica cruzada sin la complejidad de gestionar múltiples modelos aislados para texto, visión y sonido.